# Phase 2 inspectable audit

This companion reproduces bounded checks from generated mapping/coverage files. Full executed evidence is in phase2_validation.json and the scripts. These cells are supplied unexecuted, with no fabricated notebook output. Run from the repository root. National coverage is not observed; recommendation C remains.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import duckdb
root = Path.cwd()
assert (root/'results/feasibility/phase2_validation.json').exists()
out = root/'results/feasibility'
print(json.loads((out/'phase2_validation.json').read_text())['scope'])


In [ ]:
mapping = pd.read_csv(out/'occupation_mapping_final.csv', dtype={'occp':str})
assert mapping.occp.is_unique and len(mapping)==529
print(mapping.mapping_status.value_counts())
coverage = pd.read_csv(out/'exposure_coverage_by_geography.csv', dtype={'geography_id':str})
assert coverage.loc[coverage.scope=='National','weighted_coverage_percent'].isna().all()
coverage.loc[(coverage.scope=='State') & (coverage.survey_year=='pooled'), ['age_group','employment_weight','covered_weight','weighted_coverage_percent','exposure_lower','exposure_upper']]


In [ ]:
connection=duckdb.connect()
connection.execute("SET memory_limit='2GB'; SET threads=2; SET enable_external_access=false; SET allow_persistent_secrets=false")
aggregates=pd.read_csv(root/'data/feasibility/phase2_pums_occupation_aggregates.csv', dtype={'ST':str,'PUMA':str,'survey_year':str,'OCCP':str,'SOCP':str})
connection.register('aggregates',aggregates)
result=connection.execute('SELECT age_segment,survey_year,sum(weight) AS employment_weight FROM aggregates GROUP BY ALL ORDER BY ALL').df()
connection.close()
result


Executed processing SQL (prior to aggregation):

```sql
SELECT ST,PUMA,survey_year,age_segment,OCCP,SOCP,count(*) AS "rows",sum(weight) AS weight FROM chunk GROUP BY ALL ORDER BY ALL
```

The processing script checks every chunk sum against independent Python arithmetic. validate_phase2.py independently reads the complete DE archive person by person with csv.DictReader and verifies both age bands, all five years, pooled denominators, scores and conservative bounds.
